# Treino do modelo - Part2
Notebook para carregar o dataset de landmarks da mão, treinar um classificador simples, avaliar e guardar o modelo.

In [ ]:
# Imports
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [ ]:
# Caminhos
DATA_PATH = 'hand_landmarks_dataset.csv'  # ficheiro na mesma pasta
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)
print('Dataset:', DATA_PATH)
print('Model output dir:', MODELS_DIR)

In [ ]:
# Carregar dados
df = pd.read_csv(DATA_PATH)
print('shape:', df.shape)
display(df.head())
print('\nClasses (contagem):')
print(df['label'].value_counts())

**Nota sobre o CSV de landmarks**:

O ficheiro CSV gerado por `create_dataset.py` contém agora a coluna `hand` que indica `Left` ou `Right` para a mão detetada. Ao construir a matriz de características `X` no notebook, esta coluna é ignorada (remoção por `df.drop(columns=['label','hand'], errors='ignore')`).


In [ ]:
# Mostrar amostras de baixa confiança e imagem de confusão re-treinada
import os
import pandas as pd
low_path = os.path.join('models', 'low_confidence_samples.csv')
if os.path.exists(low_path):
    low_df = pd.read_csv(low_path)
    print('Low confidence samples:', len(low_df))
    display(low_df.head())
else:
    print('Nenhum ficheiro low_confidence_samples.csv encontrado em models/')

# Mostrar imagem de confusão re-treinada se existir
img_path = os.path.join('models', 'confusion_retrain_light.png')
if os.path.exists(img_path):
    from IPython.display import Image, display as _display
    _display(Image(filename=img_path))
else:
    print('Imagem de confusão re-treinada não encontrada em models/')


In [ ]:
# Preparar X e y
X = df.drop(columns=['label','hand'], errors='ignore').values
y = df['label'].values
print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# Dividir treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
# Escalar características (opcional, bom para alguns modelos)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
# Treinar um classificador simples
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train_s, y_train)
y_pred = clf.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy test: {acc:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Guardar modelo e scaler
model_path = os.path.join(MODELS_DIR, 'rf_hand_sign.pkl')
scaler_path = os.path.join(MODELS_DIR, 'scaler_hand_sign.pkl')
joblib.dump(clf, model_path)
joblib.dump(scaler, scaler_path)
print('Modelo guardado em', model_path)
print('Scaler guardado em', scaler_path)

**Próximos passos**:
- Experimentar outros modelos (SVM, XGBoost, redes neurais).
- Fazer validação cruzada e afinar hiperparâmetros (GridSearch/RandomizedSearch).
- Implementar pipeline para pré-processamento, augmentação e persistência do pipeline completo.

**Fase 2 — Treino, comparação de modelos e afinação (Resumo)**

Nesta secção do notebook vamos:

- Codificar as etiquetas (`LabelEncoder`) para treino.
- Comparar modelos base: RandomForest, SVM (com `probability=True`), KNN e XGBoost (se disponível).
- Fazer uma pesquisa rápida de hiperparâmetros (GridSearchCV) para RandomForest e SVM.
- Selecionar o melhor modelo com base na pontuação de validação e guardar o modelo final em `models/best_model.pkl`, o `LabelEncoder` em `models/label_encoder.pkl` e o `scaler` em `models/scaler_hand_sign.pkl`.

Observação: ajustar as grelhas/tempos de treino conforme capacidade de CPU. Se quiser, posso executar o treino aqui (leva algum tempo).

In [ ]:
# Encoding das labels e divisão com validação
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score

le = LabelEncoder()
y_enc = le.fit_transform(y)
print('Labels encoded:', list(le.classes_))

# Guardar cópia de treino/teste já feita anteriormente; se precisar, refazer
X_train_full, X_val, y_train_full, y_val = train_test_split(X_train_s, y_train, test_size=0.15, random_state=42, stratify=y_train)
print('Training full:', X_train_full.shape, 'Validation:', X_val.shape)


In [ ]:
# Comparação rápida de modelos base
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier()
}

results = []
for name, m in models.items():
    try:
        m.fit(X_train_full, y_train_full)
        y_pred = m.predict(X_val)
        acc = accuracy_score(y_val, y_pred)
        results.append((name, acc))
        print(f"{name}: {acc:.4f}")
    except Exception as e:
        print(f"Erro a treinar {name}: {e}")

# Se XGBoost estiver disponível, testar também
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
    xgb.fit(X_train_full, y_train_full)
    acc = accuracy_score(y_val, xgb.predict(X_val))
    results.append(('XGBoost', acc))
    print(f"XGBoost: {acc:.4f}")
except Exception:
    print('XGBoost não disponível ou erro durante treino.' )

# Ordenar resultados
results_sorted = sorted(results, key=lambda x: x[1], reverse=True)
print('\nRanking:')
for r in results_sorted: print(r)


In [ ]:
# GridSearch rápido para RandomForest e SVM (executar se tiver tempo)
from sklearn.model_selection import GridSearchCV

best_models = {}

# RF grid (pequena grelha para tempo razoável)
rf_params = {'n_estimators':[100,200], 'max_depth':[None, 20], 'min_samples_split':[2,5]}
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_gs = GridSearchCV(rf, rf_params, cv=3, scoring='accuracy', n_jobs=-1)
rf_gs.fit(X_train_full, y_train_full)
print('RF best:', rf_gs.best_params_, rf_gs.best_score_)
best_models['RandomForest'] = (rf_gs.best_estimator_, rf_gs.best_score_)

# SVM grid (pequena grelha)
svm_params = {'C':[0.1,1.0], 'kernel':['rbf','linear']}
svm = SVC(probability=True, random_state=42)
svm_gs = GridSearchCV(svm, svm_params, cv=3, scoring='accuracy', n_jobs=-1)
svm_gs.fit(X_train_full, y_train_full)
print('SVM best:', svm_gs.best_params_, svm_gs.best_score_)
best_models['SVM'] = (svm_gs.best_estimator_, svm_gs.best_score_)

# Escolher melhor dos gridsearch
best_name, best_score = None, -1
for k,(m,score) in best_models.items():
    if score > best_score:
        best_score = score
        best_name = k

print(f'Escolhido: {best_name} com score {best_score:.4f}')
best_model = best_models[best_name][0]


In [ ]:
# Re-treinar melhor modelo com todo o treino (train + val) e avaliar no teste final
# Unir X_train_full e X_val de volta para treino final
import numpy as _np

X_final_train = _np.vstack([X_train_full, X_val])
y_final_train = _np.concatenate([y_train_full, y_val])

best_model.fit(X_final_train, y_final_train)
# Avaliar no conjunto de teste (X_test_s foi definido antes)
y_test_pred = best_model.predict(X_test_s)
final_acc = accuracy_score(y_test, y_test_pred)
print(f'Accuracy final no conjunto de teste: {final_acc:.4f}')
print(classification_report(y_test, y_test_pred))

# Guardar artefactos no formato esperado pela API
import joblib
os.makedirs('models', exist_ok=True)
# salvar modelo
joblib.dump(best_model, 'models/best_model.pkl')
# salvar scaler (usado para transformar entrada antes do modelo)
joblib.dump(scaler, 'models/scaler_hand_sign.pkl')
# salvar label encoder que mapeia índices para letras
le_full = LabelEncoder()
le_full.fit(y)
joblib.dump(le_full, 'models/label_encoder.pkl')

print('Artefactos guardados: models/best_model.pkl, models/scaler_hand_sign.pkl, models/label_encoder.pkl')


**Avaliação detalhada e gráficos**

Nesta secção apresentamos uma análise visual e numérica do desempenho final do modelo: matriz de confusão, distribuição de confiança por classe e importâncias das features. Os gráficos ajudam a identificar classes com menor confiança e guiar medidas de correção (augmentação, coleta adicional, tuning).

In [ ]:
# Carregar artefactos e dados para análise final
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

MODELS_DIR = 'models'
model_path = os.path.join(MODELS_DIR, 'best_model.pkl')
scaler_path = os.path.join(MODELS_DIR, 'scaler_hand_sign.pkl')
encoder_path = os.path.join(MODELS_DIR, 'label_encoder.pkl')

model = joblib.load(model_path)
scaler = joblib.load(scaler_path)
le = joblib.load(encoder_path)

# Carregar dataset (usar augmentado se existir)
if os.path.exists('hand_landmarks_augmented.csv'):
    df = pd.read_csv('hand_landmarks_augmented.csv')
else:
    df = pd.read_csv('hand_landmarks_dataset.csv')

X = df.drop(columns=['label','hand'], errors='ignore').values
y = df['label'].values

# Obter labels codificados usando o LabelEncoder carregado
y_enc = le.transform(y)

# Reproduzir mesma divisão (random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
X_test_s = scaler.transform(X_test)

# Predições e probabilidades
y_pred = model.predict(X_test_s)
probas = model.predict_proba(X_test_s) if hasattr(model, 'predict_proba') else None

print('Test set size:', X_test_s.shape[0])
print('\nClassification report (test set):')
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
# Matriz de confusão (heatmap)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(14,12))
sns.heatmap(cm, cmap='magma', xticklabels=le.classes_, yticklabels=le.classes_, cbar=True)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.show()


In [ ]:
# Análise de confiança por classe
if probas is not None:
    max_conf = np.max(probas, axis=1)
    # distribuicao geral
    plt.figure(figsize=(8,4))
    sns.histplot(max_conf, bins=30, kde=True)
    plt.xlabel('Max confidence (per sample)')
    plt.title('Distribuição de confiança das predições (test set)')
    plt.show()

    # média por classe
    avg_conf = {}
    for i,cls in enumerate(le.classes_):
        idxs = np.where(y_test==i)[0]
        if len(idxs)>0:
            avg_conf[cls] = float(max_conf[idxs].mean())
    avg_series = pd.Series(avg_conf).sort_values()
    plt.figure(figsize=(12,5))
    avg_series.plot(kind='bar')
    plt.ylabel('Average max confidence')
    plt.ylim(0,1)
    plt.title('Average prediction confidence por classe (test set)')
    plt.tight_layout()
    plt.show()

    # listar classes com média < 0.8
    low_conf = avg_series[avg_series < 0.8]
    if not low_conf.empty:
        print('Classes com média de confiança < 0.8:')
        print(low_conf)
    else:
        print('Nenhuma classe com média de confiança < 0.8')
else:
    print('Modelo não fornece probabilidades (predict_proba não disponível).')

In [ ]:
# Importâncias de features (RandomForest)
if hasattr(model, 'feature_importances_'):
    fi = model.feature_importances_
    feature_names = [f'lm{i}_{ax}' for i in range(21) for ax in ['x','y','z']]
    fi_series = pd.Series(fi, index=feature_names).sort_values(ascending=False)
    plt.figure(figsize=(8,10))
    sns.barplot(x=fi_series.values[:30], y=fi_series.index[:30])
    plt.title('Top 30 feature importances')
    plt.tight_layout()
    plt.show()
else:
    print('Modelo não tem atributo feature_importances_.')
